In [ ]:
from abc import ABC,abstractmethod
import math
class Shape(ABC):
    """图形：所有子类必须实现 area 和 perimeter"""
    @abstractmethod
    def area(self) -> float:
        """面积"""
    @abstractmethod
    def perimeter(self) -> float:
        """周长"""
    def describe(self) -> str:
        return f"{self.__class__.__name__}:面积={self.area():.2f},周长={self.perimeter():.2f}"
class Circle(Shape):
    def __init__(self,radius:float):
        if radius <= 0:
            raise ValueError("半径必须为正数")
        self.radius = radius
    def area(self) -> float:
        return math.pi * self.radius ** 2
    def perimeter(self) -> float:
        return 2 * math.pi * self.radius
class Rectangele(Shape):
    def __init__(self,w:float,h:float):
        self.w,self.h = w,h
    def area(self) -> float:return self.w * self.h
    def perimeter(self) -> float:return 2 * (self.w + self.h)
c = Circle(5)
print(c.describe())

Circle:面积=78.54,周长=31.42


In [ ]:
class A:
    def greet(self):
        print("A")
class B(A):
    def greet(self):
        print("B")
        super().greet()
class C(A):
    def greet(self):
        print("C")
        super().greet()
class D(B,C):
    def greet(self):
        print("D")
        super().greet()
D().greet()
print(D.__mro__)
print([c.__name__ for c in D.mro()])

D
B
C
A
(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)
['D', 'B', 'C', 'A', 'object']


In [3]:
class X:pass
class Y:pass
class Z(X,Y):pass
print(Z.mro())

[<class '__main__.Z'>, <class '__main__.X'>, <class '__main__.Y'>, <class 'object'>]


In [7]:
class LogMixin:
    def log(self,msg:str):
        print(f"[{self.__class__.__name__}]{msg}")
class JSONMixin:
    def to_json(self) -> str:
        import json
        data = {k:v for k,v in vars(self).items() if not k.startswith("_")}
        return json.dumps(data,ensure_ascii=False,indent=2)
class User(LogMixin,JSONMixin):
    def __init__(self,name,age):
        self.name = name
        self.age = age
u = User("小明",20)
u.log("用户创建")
print(u.to_json())

[User]用户创建
{
  "name": "小明",
  "age": 20
}


In [8]:
class Base:
    def __init__(self):
        print("Base init")
        super().__init__()
class Mixin1(Base):
    def __init__(self):
        print("Mixin1 init")
        super().__init__()
class Mixin2(Base):
    def __init__(self):
        print("Mixin2 init")
        super().__init__()
class Child(Mixin1,Mixin2):
    def __init__(self):
        print("Child init")
        super().__init__()
Child()
print([c.__name__ for c in Child.mro()])

Child init
Mixin1 init
Mixin2 init
Base init
['Child', 'Mixin1', 'Mixin2', 'Base', 'object']


In [11]:
class Multiplier:
    def __init__(self,factor):
        self.factor = factor
    def __call__(self,x):
        return x * self.factor
double = Multiplier(2)
print(double(5))
print(callable(double))
print(callable(42))
class RangeValidator:
    def __init__(self,lo,hi):
        self.lo,self.hi = lo,hi
    def __call__(self,value):
        if not (self.lo <= value <= self.hi):
            raise ValueError(f"{value}不在[{self.lo},{self.hi}]范围内")
        return True
age_check = RangeValidator(0,150)
age_check(25)

10
True
False


True

In [12]:
class TagCloud:
    def __init__(self):
        self._tags = {}
    def add(self,tag):
        tag = tag.lower()
        self._tags[tag] = self._tags.get(tag,0) + 1
    def __getitem__(self,tag):
        return self._tags.get(tag.lower(),0)
    def __setitem__(self,tag,count):
        self._tags[tag.lower()] = count
    def __delitem__(self,tag):
        if tag.lower() in self._tags:
            del self._tags[tag.lower()]
    def __len__(self):
        return len(self._tags)
    def __iter__(self):
        return iter(self._tags)
    def __contains__(self,tag):
        return tag.lower() in self._tags
    def __repr__(self):
        return f"TagCloud({self._tags})"
cloud = TagCloud()
for t in ["Python","python","Java","Python","java"]:
    cloud.add(t)
print(cloud["python"])
print(len(cloud))
print("java" in cloud)
del cloud["java"]
print(list(cloud))

3
2
True
['python']


In [13]:
class Config:
    _readonly = {"env"}
    def __init__(self,**kwargs):
        object.__setattr__(self,"_data",{})
        for k,v in kwargs.items():
            self._data[k] = v
    def __getattr__(self,name):
        if name in self._data:
            return self._data[name]
        raise AttributeError(f"配置项 '{name}' 不存在")
    def __setattr__(self,name,value):
        if name == "_data" or name.startswith("_"):
            object.__setattr__(self,name,value)
        elif name in self._readonly:
            raise PermissionError(f"'{name}'是只读配置")
        else:
            self._data[name] = value
    def __delattr__(self,name):
        if name in self._data:
            del self._data[name]
        else:
            raise AttributeError(name)
    def __repr__(self):
        return f"Config({self._data})"
cfg = Config(env="prod",port=8080,debug=False)
print(cfg.port)
cfg.debug = True
print(cfg)

8080
Config({'env': 'prod', 'port': 8080, 'debug': True})


In [14]:
class LoggedAttr:
    def __getattribute__(self,name):
        print(f"  访问 {name}")
        return object.__getattribute__(self,name)
class Point(LoggedAttr):
    def __init__(self,x,y):
        self.x = x
        self.y = y
p = Point(1,2)
print(p.x)

  访问 x
1


In [15]:
class LazyImage:
    def __init__(self,path):
        self.path = path
        self._data = None
    @property
    def data(self):
        if self._data is None:
            print(f"  [懒加载] 读取 {self.path}")
            self._data = f"<图像数据:{self.path}>"
        return self._data
img = LazyImage("photo.png")
print("创建完毕，还没读文件")
print(img.data)
print(img.data)

创建完毕，还没读文件
  [懒加载] 读取 photo.png
<图像数据:photo.png>
<图像数据:photo.png>


In [16]:
class SingletonMeta(type):
    _instances = {}
    def __call__(cls,*args,**kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args,**kwargs)
        return cls._instances[cls]
class Logger(metaclass=SingletonMeta):
    def __init__(self):
        self.logs = []
    def info(self,msg):
        self.logs.append(f"[INFO] {msg}")
a = Logger(); b = Logger()
print(a is b)
a.info("启动");b.info("就绪")
print(a.logs)

True
['[INFO] 启动', '[INFO] 就绪']
